In [7]:
import numpy as np
import torch
import torch.nn as nn
import math
import time
import os
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
if device=="cuda":
    print(torch.cuda.get_device_name(0))

NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [91]:
data = ""
from datasets import load_dataset

ds = load_dataset("OpenRL/daily_dialog")

print(ds)
for conversation in ds["train"]:
    for message in conversation["dialog"]:
        data += message + "\n"

data = data.strip()

print("Total characters:", len(data))
print(data[:1000])

DatasetDict({
    train: Dataset({
        features: ['dialog', 'act', 'emotion'],
        num_rows: 11118
    })
    validation: Dataset({
        features: ['dialog', 'act', 'emotion'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['dialog', 'act', 'emotion'],
        num_rows: 1000
    })
})
Total characters: 5482279
Say , Jim , how about going for a few beers after dinner ? 
 You know that is tempting but is really not good for our fitness . 
 What do you mean ? It will help us to relax . 
 Do you really think so ? I don't . It will just make us fat and act silly . Remember last time ? 
 I guess you are right.But what shall we do ? I don't feel like sitting at home . 
 I suggest a walk over to the gym where we can play singsong and meet some of our friends . 
 That's a good idea . I hear Mary and Sally often go there to play pingpong.Perhaps we can make a foursome with them . 
 Sounds great to me ! If they are willing , we could ask them to go dancing with us

In [94]:
chars = sorted(set(data))
vocab_size = len(chars)
char_to_idx = {c:i for i, c in enumerate(chars)} 
idx_to_char = {i:c for i, c in enumerate(chars)} 


In [95]:
def encode(s):
    return [char_to_idx[c] for c in s]

def decode(s):
    return "".join([idx_to_char[i] for i in s])

print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {''.join(chars)}")
print(f"\nExample encoding:")
print(f"  'Hello' -> {encode('Hello')}")
print(f"  {encode('Hello')} -> '{decode(encode('Hello'))}'")

Vocabulary size: 100
Characters: 
 !"#$%&'()*+,-./0123456789:;=?@ABCDEFGHIJKLMNOPQRSTUVWXYZ\_abcdefghijklmnopqrstuvwxyz~£¥°–—‘’“”′、。

Example encoding:
  'Hello' -> [39, 64, 71, 71, 74]
  [39, 64, 71, 71, 74] -> 'Hello'


In [96]:
# Encoding
t = torch.tensor(encode(data), dtype=torch.long)

In [97]:
# Train Test split
TRAIN_LEN = int(len(t)*0.9)
VAL_LEN = int(len(t)*0.1//1)
X_train = t[:TRAIN_LEN]
X_val = t[TRAIN_LEN:]

In [98]:
def get_batch(split, batch_size, context_length):
    d = X_train if split=="train" else X_val
    ix = torch.randint(len(d)-context_length, (batch_size,)) 
    x = torch.stack([d[i:i+context_length] for i in ix])
    y = torch.stack([d[i+1:i+context_length+1] for i in ix])
    return x.to(device), y.to(device)

# Quick test
xb, yb = get_batch("train", batch_size=4, context_length=8)
print(f"Input shape:  {xb.shape}  (batch_size x context_length)")
print(f"Target shape: {yb.shape}")
print(f"\nExample (first sequence):")
print(f"  Input:  {decode(xb[0].tolist())!r}")
print(f"  Target: {decode(yb[0].tolist())!r}")
print(f"  (Target is input shifted by 1 character)")

Input shape:  torch.Size([4, 8])  (batch_size x context_length)
Target shape: torch.Size([4, 8])

Example (first sequence):
  Input:  ', you ar'
  Target: ' you are'
  (Target is input shifted by 1 character)


In [99]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim)) # it is a learnable parameter which weights can be changes

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x/rms)*self.weight

In [100]:

# We will also use Dropout throughout the model.
# Dropout randomly zeroes some values during training,
# forcing the model to not rely on any single feature.
# This prevents memorization (overfitting).
DROPOUT = 0.2


# --- Demo ---
demo_x = torch.randn(2, 4, 8)
norm = RMSNorm(8)
demo_out = norm(demo_x)

print("RMSNorm demo:")
print(f"  Input  - mean: {demo_x.mean():.3f}, std: {demo_x.std():.3f}")
print(f"  Output - mean: {demo_out.mean():.3f}, std: {demo_out.std():.3f}")
print(f"  Input range:  [{demo_x.min():.3f}, {demo_x.max():.3f}]")
print(f"  Output range: [{demo_out.min():.3f}, {demo_out.max():.3f}]")
print(f"\n  Parameters: just a scale vector of size {norm.weight.shape}")

RMSNorm demo:
  Input  - mean: -0.110, std: 1.004
  Output - mean: -0.088, std: 1.004
  Input range:  [-2.083, 2.847]
  Output range: [-2.397, 2.372]

  Parameters: just a scale vector of size torch.Size([8])


In [101]:
#Section 2: Rotary Positional Embeddings (RoPE)
#What you know: Sinusoidal positional encoding — add a fixed pattern to embeddings.
#The upgrade: RoPE rotates Q and K vectors based on position. The dot product then naturally captures relative distance between tokens.
#Like a clock: the angle between 3 o'clock and 5 o'clock = the angle between 7 o'clock and 9 o'clock. Absolute position doesn't matter, only the gap.

In [102]:
def precompute_rope_freqs(head_dim, max_seq_len, base=10000.0):
    """
    Precompute cosine and sine tables for RoPE.

    Each pair of dimensions gets a different rotation frequency.
    Low dims  -> fast rotation -> short-range patterns
    High dims -> slow rotation -> long-range patterns
    """

    freqs = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))    # 1, 0.1, 0.01, 0.001 
    # [0, 2, 4, 6]
    # [0, 0.25, 0.5, 0.75]
    # 10000^[0, 0.25, 0.5, 0.75]
    # 1 / [1, 10, 100, 1000]
    # 1, 0.1, 0.01, 0.001

    positions = torch.arange(max_seq_len).float()
    # If max_seq_len = 4:
    # torch.arange(4) -> [0, 1, 2, 3]
    # .float()        -> [0.0, 1.0, 2.0, 3.0]

    """
                        1      0.1      0.01      0.001
                    ┌──────┬────────┬─────────┬─────────┐
            pos 0   │  0   │   0    │    0    │    0    │
            pos 1   │  1   │  0.1   │   0.01  │  0.001  │
            pos 2   │  2   │  0.2   │   0.02  │  0.002  │
            pos 3   │  3   │  0.3   │   0.03  │  0.003  │
                    └──────┴────────┴─────────┴─────────┘
    """

    angles = torch.outer(positions, freqs)     
           # Multiply
    return torch.cos(angles), torch.sin(angles)       # RETURN 

In [103]:
def apply_rope(x, cos, sin):
    """
        Apply rotary embeddings to a tensor.
        x: [batch, n_heads, seq_len, head_dim]
        cos, sin: [seq_len, head_dim // 2]

        For each pair of dimensions (2i, 2i+1):
        rotated_2i   = x_2i * cos - x_2i+1 * sin
        rotated_2i+1 = x_2i * sin + x_2i+1 * cos
    """
    
    seq_len = x.shape[2]
    cos = cos[:seq_len].unsqueeze(0).unsqueeze(0)  # [1, 1, seq, hd//2]
    sin = sin[:seq_len].unsqueeze(0).unsqueeze(0)  # [1, 1, seq, hd//2]
    x1 = x[..., ::2]                               # even dims
    x2 = x[..., 1::2]                              # odd dims
    out1 = x1 * cos - x2 * sin                     
    out2 = x1 * sin + x2 * cos
    return torch.stack([out1, out2], dim=-1).flatten(-2)


    """
        seq=8
        cos = [[...8 cos val]]
        sin = [[...8 sin val]]

        out1 = [A, C]     ← rotated even dimensions
        out2 = [B, D]     ← rotated odd dimensions

                ↓ stack

        [[A, B],
        [C, D]]

                ↓ flatten

        [A, B, C, D]

        RoPE does not treat all 8 numbers together.
        Because a pair of numbers can represent a 2D point.
        Eg.     (x,y)=(3,4)

        x=xcosθ−ysinθ
        y=xsinθ+ycosθ   

    """

In [104]:
def repeat_kv(x, n_rep):
    """
    Repeat KV heads to match the number of query heads.
    x: [batch, n_kv_heads, seq_len, head_dim]
    Returns: [batch, n_kv_heads * n_rep, seq_len, head_dim]
    """
    if n_rep == 1:
        return x
    b, n_kv, seq, hd = x.shape
    return (x[:, :, None, :, :]
            .expand(b, n_kv, n_rep, seq, hd)
            .reshape(b, n_kv * n_rep, seq, hd))

"""
x = [
  [
    [ ["A", "B"], ["C", "D"] ],    ← KV head 0
    [ ["E", "F"], ["G", "H"] ]     ← KV head 1
  ]
]


## x[:, :, None, :, :]

x[:, :, None, :, :]

[
  [
    [
      [ ["A","B"], ["C","D"] ]
    ],                              ← KV head 0

    [
      [ ["E","F"], ["G","H"] ]
    ]                               ← KV head 1
  ]
]

## expand(b, n_kv, n_rep, seq, hd)

.expand(1, 2, 2, 2, 2)

[
  [
    [ ["A","B"], ["C","D"] ],   ← repetition 0
    [ ["A","B"], ["C","D"] ]    ← repetition 1
  ],

  [
    [ ["E","F"], ["G","H"] ],   ← repetition 0
    [ ["E","F"], ["G","H"] ]    ← repetition 1
  ]
]

## .reshape(1, 4, 2, 2)

[
  [
    [ ["A","B"], ["C","D"] ],   ← KV0
    [ ["A","B"], ["C","D"] ],   ← KV0 repeated
    [ ["E","F"], ["G","H"] ],   ← KV1
    [ ["E","F"], ["G","H"] ]    ← KV1 repeated
  ]
]

"""

'\nx = [\n  [\n    [ ["A", "B"], ["C", "D"] ],    ← KV head 0\n    [ ["E", "F"], ["G", "H"] ]     ← KV head 1\n  ]\n]\n\n\n## x[:, :, None, :, :]\n\nx[:, :, None, :, :]\n\n[\n  [\n    [\n      [ ["A","B"], ["C","D"] ]\n    ],                              ← KV head 0\n\n    [\n      [ ["E","F"], ["G","H"] ]\n    ]                               ← KV head 1\n  ]\n]\n\n## expand(b, n_kv, n_rep, seq, hd)\n\n.expand(1, 2, 2, 2, 2)\n\n[\n  [\n    [ ["A","B"], ["C","D"] ],   ← repetition 0\n    [ ["A","B"], ["C","D"] ]    ← repetition 1\n  ],\n\n  [\n    [ ["E","F"], ["G","H"] ],   ← repetition 0\n    [ ["E","F"], ["G","H"] ]    ← repetition 1\n  ]\n]\n\n## .reshape(1, 4, 2, 2)\n\n[\n  [\n    [ ["A","B"], ["C","D"] ],   ← KV0\n    [ ["A","B"], ["C","D"] ],   ← KV0 repeated\n    [ ["E","F"], ["G","H"] ],   ← KV1\n    [ ["E","F"], ["G","H"] ]    ← KV1 repeated\n  ]\n]\n\n'

In [ ]:
def repeat_kv(x, n_rep):
    """
    Repeat KV heads to match the number of query heads.
    x: [batch, n_kv_heads, seq_len, head_dim]
    Returns: [batch, n_kv_heads * n_rep, seq_len, head_dim]
    """
    if n_rep == 1:
        return x
    b, n_kv, seq, hd = x.shape
    return (x[:, :, None, :, :]
            .expand(b, n_kv, n_rep, seq, hd)
            .reshape(b, n_kv * n_rep, seq, hd))


class GroupedQueryAttention(nn.Module):
    """
    Grouped Query Attention with RoPE.
    n_heads query heads, n_kv_heads key/value heads.
    Groups of (n_heads // n_kv_heads) query heads share one KV pair.
    """
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0 

        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(n_heads * self.head_dim, d_model, bias=False)

    def forward(self, x, rope_cos, rope_sin):
        b, seq, _ = x.shape

        # Project Q, K, V
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # Reshape into heads [2, 10, 512] -> [2, 10, 8, 64]
        q = q.view(b, seq, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(b, seq, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = v.view(b, seq, self.n_kv_heads, self.head_dim).transpose(1, 2)

        # Apply RoPE to Q and K (not V!)
        q = apply_rope(q, rope_cos, rope_sin)
        k = apply_rope(k, rope_cos, rope_sin)

        # Repeat KV heads to match Q heads
        k = repeat_kv(k, self.n_rep)
        v = repeat_kv(v, self.n_rep)

        # Scaled dot-product attention with causal mask
        scale = 1.0 / math.sqrt(self.head_dim)
        scores = (q @ k.transpose(-2, -1)) * scale

        mask = torch.triu(torch.ones(seq, seq, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-inf"))

        weights = F.softmax(scores, dim=-1)

        # Dropout on attention weights (regularization)
        weights = F.dropout(weights, p=DROPOUT, training=self.training)

        out = weights @ v

        # Merge heads and project
        out = out.transpose(1, 2).contiguous().view(b, seq, -1)
        return self.o_proj(out) 

In [106]:
class SwiGLU(nn.Module):
    """
    SwiGLU Feed-Forward Network.

    Two paths:
      gate: SiLU(x @ W_gate) - controls flow
      up:   x @ W_up         - carries information

    Combined: gate * up -> W_down

    SiLU(x) = x * sigmoid(x), a smooth version of ReLU.
    """
    
    def __init__(self, d_model, hidden_dim):
        super().__init__()
        self.w_gate = nn.Linear(d_model, hidden_dim, bias=False)
        self.w_up   = nn.Linear(d_model, hidden_dim, bias=False)
        self.w_down = nn.Linear(hidden_dim, d_model, bias=False)

    def forward(self, x):
        gate = F.silu(self.w_gate(x))
        up   = self.w_up(x)
        return F.dropout(self.w_down(gate * up), p=DROPOUT, training=self.training)

In [107]:
class TransformerBlock(nn.Module):
    """
    One layer of a modern transformer.

    Pre-norm architecture:
      x -> RMSNorm -> GQA Attention -> + residual
      x -> RMSNorm -> SwiGLU FFN     -> + residual
    """
    def __init__(self, d_model, n_heads, n_kv_heads, ffn_hidden_dim):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attention = GroupedQueryAttention(d_model, n_heads, n_kv_heads)
        self.ffn_norm  = RMSNorm(d_model)
        self.ffn       = SwiGLU(d_model, ffn_hidden_dim)

    def forward(self, x, rope_cos, rope_sin):
        x = x + self.attention(self.attn_norm(x), rope_cos, rope_sin)
        x = x + self.ffn(self.ffn_norm(x))
        return x

In [108]:

class MiniLLM(nn.Module):
    """
    A small but modern language model.

    Architecture: modern transformer with all 4 upgrades.
    Training objective: next character prediction.
    """
    def __init__(self, vocab_size, d_model, n_layers, n_heads, n_kv_heads,
                 ffn_hidden_dim, max_seq_len):
        super().__init__()

        self.d_model = d_model
        self.max_seq_len = max_seq_len

        # Token embedding (no positional embedding -- RoPE handles position)
        self.token_emb = nn.Embedding(vocab_size, d_model)

        # Transformer blocks
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads, n_kv_heads, ffn_hidden_dim)
            for _ in range(n_layers)
        ])

        # Final norm and output head
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying: share embedding and output weights
        self.lm_head.weight = self.token_emb.weight

        # Precompute RoPE frequencies
        head_dim = d_model // n_heads
        rope_cos, rope_sin = precompute_rope_freqs(head_dim, max_seq_len)
        self.register_buffer("rope_cos", rope_cos)
        self.register_buffer("rope_sin", rope_sin)

        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        b, seq_len = idx.shape

        # Token embedding
        x = self.token_emb(idx)

        # Pass through transformer blocks
        for layer in self.layers:
            x = layer(x, self.rope_cos, self.rope_sin)

        # Final norm + project to vocabulary
        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )

        return logits, loss

In [109]:
# --- Model Configuration ---

config = {
    "vocab_size":     vocab_size,
    "d_model":        256,
    "n_layers":       4,
    "n_heads":        8,
    "n_kv_heads":     2,
    "ffn_hidden_dim": 680,
    "max_seq_len":    256,
}

model = MiniLLM(**config).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 50)
print("  MODEL SUMMARY")
print("=" * 50)
print(f"  Vocabulary:      {config['vocab_size']}")
print(f"  Embedding dim:   {config['d_model']}")
print(f"  Layers:          {config['n_layers']}")
print(f"  Query heads:     {config['n_heads']}")
print(f"  KV heads:        {config['n_kv_heads']} (GQA ratio: {config['n_heads']//config['n_kv_heads']}:1)")
print(f"  FFN hidden dim:  {config['ffn_hidden_dim']}")
print(f"  Context length:  {config['max_seq_len']}")
print(f"  Head dim:        {config['d_model'] // config['n_heads']}")
print(f"{'=' * 50}")
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size (approx):  {total_params * 4 / 1e6:.1f} MB (float32)")
print(f"{'=' * 50}")

  MODEL SUMMARY
  Vocabulary:      100
  Embedding dim:   256
  Layers:          4
  Query heads:     8
  KV heads:        2 (GQA ratio: 4:1)
  FFN hidden dim:  680
  Context length:  256
  Head dim:        32
  Total parameters:     2,772,224
  Trainable parameters: 2,772,224
  Model size (approx):  11.1 MB (float32)


In [110]:
# --- Training Hyperparameters ---
BATCH_SIZE = 64
CONTEXT_LEN = config["max_seq_len"]
LEARNING_RATE = 3e-4
MAX_STEPS = 3000
EVAL_INTERVAL = 250
EVAL_STEPS = 20
LOG_INTERVAL = 50

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

In [111]:
@torch.no_grad()
def estimate_loss():
    """Estimate loss on train and val splits."""
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = []
        for _ in range(EVAL_STEPS):
            xb, yb = get_batch(split, BATCH_SIZE, CONTEXT_LEN)
            _, loss = model(xb, yb)
            losses.append(loss.item())
        out[split] = sum(losses) / len(losses)
    model.train()
    return out

In [112]:
# --- Training Loop ---
print("Starting training...")
print(f"  {MAX_STEPS} steps, batch_size={BATCH_SIZE}, context_len={CONTEXT_LEN}")
print(f"  Evaluating every {EVAL_INTERVAL} steps")
print("-" * 60)

train_losses = []
val_losses = []
step_log = []
start_time = time.time()

model.train()
for step in range(MAX_STEPS):
    xb, yb = get_batch("train", BATCH_SIZE, CONTEXT_LEN)

    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    if step % LOG_INTERVAL == 0:
        elapsed = time.time() - start_time
        print(f"  Step {step:5d}/{MAX_STEPS} | Loss: {loss.item():.4f} | Time: {elapsed:.0f}s")

    if step % EVAL_INTERVAL == 0 or step == MAX_STEPS - 1:
        losses = estimate_loss()
        train_losses.append(losses["train"])
        val_losses.append(losses["val"])
        step_log.append(step)
        if step > 0:
            elapsed = time.time() - start_time
            steps_per_sec = step / elapsed
            remaining = (MAX_STEPS - step) / steps_per_sec
            print(f"  >>> Eval @ step {step}: train={losses['train']:.4f}, val={losses['val']:.4f} | ~{remaining:.0f}s remaining")

total_time = time.time() - start_time
print("-" * 60)
print(f"Training complete! Total time: {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"Final train loss: {train_losses[-1]:.4f}")
print(f"Final val loss:   {val_losses[-1]:.4f}")

Starting training...
  3000 steps, batch_size=64, context_len=256
  Evaluating every 250 steps
------------------------------------------------------------
  Step     0/3000 | Loss: 4.6361 | Time: 0s
  Step    50/3000 | Loss: 2.4292 | Time: 17s
  Step   100/3000 | Loss: 2.0380 | Time: 30s
  Step   150/3000 | Loss: 1.7528 | Time: 43s
  Step   200/3000 | Loss: 1.5930 | Time: 57s
  Step   250/3000 | Loss: 1.5475 | Time: 70s
  >>> Eval @ step 250: train=1.4349, val=1.4394 | ~814s remaining
  Step   300/3000 | Loss: 1.4508 | Time: 87s
  Step   350/3000 | Loss: 1.3900 | Time: 100s
  Step   400/3000 | Loss: 1.3615 | Time: 113s
  Step   450/3000 | Loss: 1.3194 | Time: 127s
  Step   500/3000 | Loss: 1.3028 | Time: 140s
  >>> Eval @ step 500: train=1.2351, val=1.2426 | ~720s remaining
  Step   550/3000 | Loss: 1.3053 | Time: 157s
  Step   600/3000 | Loss: 1.2789 | Time: 170s
  Step   650/3000 | Loss: 1.2208 | Time: 184s
  Step   700/3000 | Loss: 1.2508 | Time: 197s
  Step   750/3000 | Loss: 1.22

In [113]:
# Save the trained model
torch.save(model.state_dict(), "model.pt")

print("Model saved successfully!")

Model saved successfully!


In [114]:

"""
↓
Q projection
K projection
V projection
 ↓
RoPE
 ↓
QKᵀ / √d
 ↓
Causal mask
 ↓
Softmax
 ↓
Attention × V
 ↓
Output projection
"""


'\n↓\nQ projection\nK projection\nV projection\n ↓\nRoPE\n ↓\nQKᵀ / √d\n ↓\nCausal mask\n ↓\nSoftmax\n ↓\nAttention × V\n ↓\nOutput projection\n'

In [119]:
# --- Model Configuration ---

config = {
    "vocab_size":     vocab_size,
    "d_model":        256,
    "n_layers":       4,
    "n_heads":        8,
    "n_kv_heads":     2,
    "ffn_hidden_dim": 680,
    "max_seq_len":    256,
}

model = MiniLLM(**config).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 50)
print("  MODEL SUMMARY")
print("=" * 50)
print(f"  Vocabulary:      {config['vocab_size']}")
print(f"  Embedding dim:   {config['d_model']}")
print(f"  Layers:          {config['n_layers']}")
print(f"  Query heads:     {config['n_heads']}")
print(f"  KV heads:        {config['n_kv_heads']} (GQA ratio: {config['n_heads']//config['n_kv_heads']}:1)")
print(f"  FFN hidden dim:  {config['ffn_hidden_dim']}")
print(f"  Context length:  {config['max_seq_len']}")
print(f"  Head dim:        {config['d_model'] // config['n_heads']}")
print(f"{'=' * 50}")
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size (approx):  {total_params * 4 / 1e6:.1f} MB (float32)")
print(f"{'=' * 50}")

  MODEL SUMMARY
  Vocabulary:      100
  Embedding dim:   256
  Layers:          4
  Query heads:     8
  KV heads:        2 (GQA ratio: 4:1)
  FFN hidden dim:  680
  Context length:  256
  Head dim:        32
  Total parameters:     2,772,224
  Trainable parameters: 2,772,224
  Model size (approx):  11.1 MB (float32)


In [120]:
# --- Training Hyperparameters ---
BATCH_SIZE = 64
CONTEXT_LEN = config["max_seq_len"]
LEARNING_RATE = 3e-4
MAX_STEPS = 3000
EVAL_INTERVAL = 250
EVAL_STEPS = 20
LOG_INTERVAL = 50
VOCAB_SIZE = 65
D_MODEL = 256
N_LAYERS = 4
N_HEADS = 8
N_KV_HEADS = 2
FFN_HIDDEN_DIM = 680
MAX_SEQ_LEN = 256

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

In [121]:
@torch.no_grad()
def estimate_loss():
    """Estimate loss on train and val splits."""
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = []
        for _ in range(EVAL_STEPS):
            xb, yb = get_batch(split, BATCH_SIZE, CONTEXT_LEN)
            _, loss = model(xb, yb)
            losses.append(loss.item())
        out[split] = sum(losses) / len(losses)
    model.train()
    return out

In [132]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# IMPORTANT: this must be the SAME vocabulary used during training
vocab_size = len(chars)

model = MiniLLM(
    vocab_size=vocab_size,
    d_model=256,
    n_layers=4,
    n_heads=8,
    n_kv_heads=2,
    ffn_hidden_dim=680,
    max_seq_len=256
)

# Load trained weights
checkpoint = torch.load(
    "model.pt",
    map_location="cpu"
)

model.load_state_dict(checkpoint)

# NOW move the complete model to GPU
model = model.to(device)

model.eval()

MiniLLM(
  (token_emb): Embedding(100, 256)
  (layers): ModuleList(
    (0-3): 4 x TransformerBlock(
      (attn_norm): RMSNorm()
      (attention): GroupedQueryAttention(
        (q_proj): Linear(in_features=256, out_features=256, bias=False)
        (k_proj): Linear(in_features=256, out_features=64, bias=False)
        (v_proj): Linear(in_features=256, out_features=64, bias=False)
        (o_proj): Linear(in_features=256, out_features=256, bias=False)
      )
      (ffn_norm): RMSNorm()
      (ffn): SwiGLU(
        (w_gate): Linear(in_features=256, out_features=680, bias=False)
        (w_up): Linear(in_features=256, out_features=680, bias=False)
        (w_down): Linear(in_features=680, out_features=256, bias=False)
      )
    )
  )
  (final_norm): RMSNorm()
  (lm_head): Linear(in_features=256, out_features=100, bias=False)
)

In [133]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load("model.pt", map_location="cpu")
for key, value in checkpoint.items():
    print(f"{key:60} {tuple(value.shape)}")

rope_cos                                                     (256, 16)
rope_sin                                                     (256, 16)
token_emb.weight                                             (100, 256)
layers.0.attn_norm.weight                                    (256,)
layers.0.attention.q_proj.weight                             (256, 256)
layers.0.attention.k_proj.weight                             (64, 256)
layers.0.attention.v_proj.weight                             (64, 256)
layers.0.attention.o_proj.weight                             (256, 256)
layers.0.ffn_norm.weight                                     (256,)
layers.0.ffn.w_gate.weight                                   (680, 256)
layers.0.ffn.w_up.weight                                     (680, 256)
layers.0.ffn.w_down.weight                                   (256, 680)
layers.1.attn_norm.weight                                    (256,)
layers.1.attention.q_proj.weight                             (256, 256)
layers.1

In [134]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=500, temperature=0.8):
    """
    Generate text autoregressively.

    temperature controls randomness:
      low (0.3)  -> conservative, repetitive
      mid (0.8)  -> balanced
      high (1.2) -> creative, chaotic
    """
    model.eval()
    tokens = encode(prompt)
    tokens = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        context = tokens[:, -config["max_seq_len"]:]
        logits, _ = model(context)
        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        tokens = torch.cat([tokens, next_token], dim=1)

    return decode(tokens[0].tolist())

In [135]:
# --- Generate at different temperatures ---
prompt = "the color of sky is"

print("=" * 60)
print(f"  PROMPT: {prompt!r}")
print("=" * 60)

for temp in [0.5, 0.8, 1.0, 1.2]:
    print(f"\n{'_' * 60}")
    print(f"  Temperature = {temp}")
    print(f"{'_' * 60}")
    output = generate(model, prompt, max_new_tokens=300, temperature=temp)
    print(output)

  PROMPT: 'the color of sky is'

____________________________________________________________
  Temperature = 0.5
____________________________________________________________
the color of sky is the last time . 
 I see . But I don't think I can transfer to confirm the ball . 
 I will . 
 What would you like to pay for ? 
 I think I'd like to . 
 What is the problem ? 
 It's a very school to find a decision . 
 It's a really problem . 
 What do you think of the price are you from ? 
 I thin

____________________________________________________________
  Temperature = 0.8
____________________________________________________________
the color of sky is pretty cheaper to the response . 
 I ’ m so assure we can buy a discount then . 
 Do you have a good preparate ? 
 yes , I like such a chicken . 
 Oh , your order bad places the bank here and you can conclude me . 
 We can ’ t be able to be interested in . 
 You ’ re right . I ’ m before I go to th

_________________________________________